# E1.8 · Third-party and model supply chain risk

**Function E — Governance, Risk, Compliance & the CISO Office → The GRC Practitioner (Risk & Control)**  ·  *Security of AI*

Builds on **[E1.7 · Continuous control verification](https://spbreed.github.io/cyber-commons/lessons/E1.7.html)**.

| | |
|---|---|
| Open-source tooling | OWASP AIBOM, Sigstore |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Third-party risk for AI has the ordinary supply-chain problem plus a question
nobody's assessment form asks:

> **Can this component change without telling us?**

For a library the answer is no — you pin a version. For a hosted model the
answer is usually yes, and it changes the risk rating, because every control you
tested was tested against behaviour the vendor can replace on a Tuesday.

Three artefact classes, with genuinely different maturity:

- **Libraries** — signing, version pinning, download signals. Mature.
- **Model weights or a hosted model** — attestation possible and rare; no
  popularity signal that means anything; version stability is a contractual
  question, not a technical one.
- **Prompt and tool packages (MCP, skills)** — no signing convention, and they
  run with your agent's authority.

Saying which signals are unavailable is part of the assessment, not a gap in it.

## 2 · Demo — the ordinary signals, and where they run out

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Component:
    name: str; kind: str; signed: bool = False
    pinned: bool = False; can_change_silently: bool = False
    runs_with_agent_authority: bool = False; downloads: int = 0

COMPONENTS = [
 Component("cryptography==42.0.5", "library", True, True, False, False, 900_000),
 Component("langchain==0.2.1", "library", False, True, False, False, 400_000),
 Component("hosted GLM-4.6 endpoint", "hosted model", False, False, True, False),
 Component("local glm-4.6 weights (pinned digest)", "weights", True, True, False, False),
 Component("mcp-jira-connector==0.0.3", "tool package", False, True, False, True, 180),
]
def assess(c):
    flags = []
    if not c.signed:                  flags.append("unsigned")
    if not c.pinned:                  flags.append("not version-pinned")
    if c.can_change_silently:         flags.append("CAN CHANGE WITHOUT NOTICE")
    if c.runs_with_agent_authority:   flags.append("runs with agent authority")
    if c.kind == "library" and c.downloads < 1000: flags.append("little scrutiny")
    tier = ("high" if c.can_change_silently or c.runs_with_agent_authority
            else "medium" if flags else "low")
    return tier, flags

print(f"{'component':40s}{'kind':14s}{'tier':8s}flags")
print("-" * 96)
for c in COMPONENTS:
    tier, flags = assess(c)
    print(f"{c.name:40s}{c.kind:14s}{tier:8s}{', '.join(flags) or '—'}")

## 3 · Where it breaks — the silent change, priced

In [ ]:
import time
now = time.time(); DAY = 86400

CONTROL_TESTS = {"SB-2": now - 20*DAY, "EV-2": now - 20*DAY, "DR-1": now - 20*DAY}
MODEL_CHANGED_AT = now - 5*DAY

print("your controls were tested against a model that changed 5 days ago:")
for cid, tested in CONTROL_TESTS.items():
    valid = tested > MODEL_CHANGED_AT
    print(f"   {cid}  tested {int((now-tested)/DAY)}d ago  "
          f"{'still valid' if valid else 'INVALIDATED by the model change'}")
invalidated = [c for c, t in CONTROL_TESTS.items() if t <= MODEL_CHANGED_AT]
print(f"\n{len(invalidated)}/{len(CONTROL_TESTS)} control tests invalidated by a "
      f"change you did not make and were not told about.")
assert invalidated

## 4 · The control — the four questions, and stating the gaps

In [ ]:
QUESTIONS = [
 ("Can this component change without notifying us?",
  "if yes, every control test has an implicit expiry tied to the vendor"),
 ("Does it execute with our agent's authority?",
  "if yes, assess it as code, not as a dependency"),
 ("Can we pin a digest, and do we?",
  "the difference between a supply chain and a subscription"),
 ("What is our exit if we stop using it?",
  "DORA Art.11 asks this directly; most AI contracts have no answer"),
]
for q, why in QUESTIONS: print(f"Q: {q}\n   → {why}\n")

SIGNALS = {
 "library":      {"signature": True, "downloads": True, "pinning": True, "lineage": True},
 "hosted model": {"signature": False, "downloads": False, "pinning": False, "lineage": False},
 "weights":      {"signature": True, "downloads": False, "pinning": True, "lineage": False},
 "tool package": {"signature": False, "downloads": False, "pinning": True, "lineage": False},
}
print(f"{'artefact class':16s}{'signals available':>20}  unavailable")
print("-" * 74)
for kind, sig in SIGNALS.items():
    have = [k for k, v in sig.items() if v]
    lack = [k for k, v in sig.items() if not v]
    print(f"{kind:16s}{f'{len(have)}/{len(sig)}':>20}  {lack or '—'}")

def assessment_statement(kind):
    sig = SIGNALS[kind]
    lack = [k for k, v in sig.items() if not v]
    return (f"{kind}: assessed on {len(sig)-len(lack)}/{len(sig)} signals. "
            f"{', '.join(lack) or 'none'} unavailable for this artefact class.")
print()
for kind in SIGNALS: print("  " + assessment_statement(kind))
print("\nThat last sentence is the deliverable. A rating that hides which signals")
print("were unavailable is a number someone will later rely on.")

## What you just proved

The hosted model and the MCP tool package both tier high — one for silent change, one for running with agent authority. The silent model change invalidates all three control tests taken before it. The signal table shows libraries with 4 of 4 signals available and hosted models with 0 of 4, and each assessment statement names what was unavailable.

## Your turn

Add "can this change without notifying us?" to your third-party assessment form. For hosted models the answer is usually yes, and it should carry an explicit control-test expiry.

---

**Next → [E1.9 · Model and agent lifecycle governance](https://spbreed.github.io/cyber-commons/lessons/E1.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*